# Notebook 16 — Model Evaluation

## Leadership and Management Book Recommendation System

### Objective

This notebook consolidates the evaluation results produced throughout the project and determines the final model architecture for the Leadership and Management Book Recommendation System.

The project contains several analytical and machine-learning components with different purposes:

1. **Enriched TF-IDF**
   - represents book content using textual metadata;
   - supports exact cosine-similarity retrieval and recommendation.

2. **Truncated SVD / Latent Semantic Analysis (LSA)**
   - reduces the sparse TF-IDF representation to 200 dimensions;
   - supports clustering, visualization, and dense numerical representation.

3. **K-Means Topic Clustering**
   - identifies broad thematic groups within the catalog;
   - provides supplementary topic metadata rather than recommendation ranking.

4. **Content-Based Recommendation System**
   - retrieves books using cosine similarity;
   - supports both book-to-book and natural-language recommendation.

5. **PyTorch Autoencoder**
   - experimentally compresses the 200-dimensional LSA representation into a 32-dimensional latent space;
   - evaluates whether neural compression preserves useful semantic structure.

6. **LLM-Ready Grounding Layer**
   - prepares retrieved books and structured metadata for grounded natural-language explanation;
   - does not replace the recommendation algorithm.

---

## Evaluation Principle

The components perform different tasks and therefore should not be compared using a single universal accuracy metric.

For example:

- clustering has no supervised target;
- recommendation has no explicit user relevance labels;
- the autoencoder predicts reconstruction rather than book relevance;
- the LLM layer is an explanation architecture rather than a predictive model.

Evaluation therefore uses metrics appropriate to each component.

### Dimensionality Reduction

- explained variance;
- cosine-similarity preservation;
- Top-10 neighbor preservation.

### Clustering

- cluster structure;
- cohesion;
- interpretability;
- thematic usefulness;
- source composition.

### Recommendation

- catalog coverage;
- recommendation similarity;
- topic diversity;
- source exposure;
- retrieval robustness;
- qualitative relevance.

### Autoencoder

- reconstruction MSE;
- improvement over an untrained baseline;
- cosine-geometry preservation;
- Top-10 neighbor preservation.

### LLM Integration

- grounding completeness;
- ranking preservation;
- anti-fabrication constraints;
- retrieval robustness;
- provider-independent operation.

---

## Evaluation Philosophy

The final system is selected according to **fitness for purpose**, not model complexity.

A more complex model is not automatically considered superior.

The preferred architecture should:

- produce semantically useful recommendations;
- preserve recommendation coverage;
- remain interpretable;
- handle incomplete metadata transparently;
- avoid unsupported predictions;
- behave safely for invalid queries;
- remain computationally practical for deployment.

The evaluation therefore focuses on determining the appropriate role of each model within the final system rather than selecting a single algorithm as the universal winner.

## 1. Load Evaluation Artifacts

Evaluation results generated in previous notebooks are loaded rather than recomputed.

This ensures that Notebook 16 acts as the project's consolidated evaluation layer and does not modify the validated models.

The artifacts cover:

- SVD dimensionality reduction;
- clustering;
- content-based recommendation;
- neural-network compression;
- natural-language retrieval and LLM grounding.

Only previously generated evaluation outputs are used for final model assessment.

In [1]:
# ============================================================
# IMPORTS AND PROJECT PATHS
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Project directories
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/Users/jannoelvero/Documents/Ironhack/Week_11/"
    "Leadership_Management_Book_Recommendation_System"
)

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

MODELS_DIR = (
    PROJECT_ROOT
    / "models"
)


print("MODEL EVALUATION — PROJECT SETUP")
print("=" * 80)

print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "Processed directory exists:",
    PROCESSED_DIR.exists()
)

print(
    "Models directory exists:",
    MODELS_DIR.exists()
)

MODEL EVALUATION — PROJECT SETUP
Project root: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System
Processed directory exists: True
Models directory exists: True


In [2]:
# ============================================================
# DISCOVER MODEL EVALUATION ARTIFACTS
# ============================================================

evaluation_keywords = [
    "svd",
    "cluster",
    "recommendation",
    "autoencoder",
    "llm",
    "pytorch"
]


evaluation_artifacts = sorted([
    path
    for path in list(PROCESSED_DIR.glob("*.csv"))
    if any(
        keyword in path.name.lower()
        for keyword in evaluation_keywords
    )
])


print("AVAILABLE MODEL EVALUATION ARTIFACTS")
print("=" * 80)

print(
    "Artifacts found:",
    len(evaluation_artifacts)
)

print()

for path in evaluation_artifacts:
    print(
        path.relative_to(PROJECT_ROOT)
    )

AVAILABLE MODEL EVALUATION ARTIFACTS
Artifacts found: 19

data/processed/autoencoder_data_split.csv
data/processed/autoencoder_evaluation_summary.csv
data/processed/autoencoder_latent_32.csv
data/processed/autoencoder_training_history.csv
data/processed/books_with_final_topic_clusters.csv
data/processed/core_vs_enriched_recommendations.csv
data/processed/final_cluster_representative_books.csv
data/processed/final_topic_cluster_summary.csv
data/processed/llm_grounding_catalog.csv
data/processed/llm_retrieval_diagnostic.csv
data/processed/llm_retrieval_robustness.csv
data/processed/pytorch_tensor_book_index.csv
data/processed/recommendation_similarity_by_rank.csv
data/processed/recommendation_similarity_evaluation.csv
data/processed/recommendation_source_exposure.csv
data/processed/recommendation_topic_diversity.csv
data/processed/svd_component_search.csv
data/processed/svd_neighbor_preservation.csv
data/processed/svd_similarity_preservation.csv


## 2. Evaluation Artifact Inspection

The evaluation artifacts produced by the previous modeling notebooks are inspected before consolidation.

This step verifies:

- dataset dimensions;
- available evaluation metrics;
- column names and data types;
- consistency of previously saved results.

The objective is to use the original validated outputs directly rather than manually reproducing or re-entering evaluation statistics.

The inspection focuses initially on the primary quantitative evaluation artifacts for:

1. SVD / LSA dimensionality reduction;
2. recommendation performance;
3. autoencoder performance;
4. natural-language retrieval robustness.

Clustering evaluation will subsequently be incorporated using its cluster-level outputs because its assessment includes both quantitative structure and semantic interpretability.

In [3]:
# ============================================================
# LOAD AND INSPECT PRIMARY EVALUATION ARTIFACTS
# ============================================================

evaluation_files = {
    "SVD Component Search":
        PROCESSED_DIR / "svd_component_search.csv",

    "SVD Neighbor Preservation":
        PROCESSED_DIR / "svd_neighbor_preservation.csv",

    "SVD Similarity Preservation":
        PROCESSED_DIR / "svd_similarity_preservation.csv",

    "Recommendation Evaluation":
        PROCESSED_DIR / "recommendation_similarity_evaluation.csv",

    "Recommendation Diversity":
        PROCESSED_DIR / "recommendation_topic_diversity.csv",

    "Recommendation Source Exposure":
        PROCESSED_DIR / "recommendation_source_exposure.csv",

    "Autoencoder Evaluation":
        PROCESSED_DIR / "autoencoder_evaluation_summary.csv",

    "LLM Retrieval Diagnostic":
        PROCESSED_DIR / "llm_retrieval_diagnostic.csv",

    "LLM Retrieval Robustness":
        PROCESSED_DIR / "llm_retrieval_robustness.csv",

    "Cluster Summary":
        PROCESSED_DIR / "final_topic_cluster_summary.csv"
}


evaluation_data = {}


print("PRIMARY EVALUATION ARTIFACT INSPECTION")
print("=" * 100)


for name, path in evaluation_files.items():

    print(f"\n{name}")
    print("-" * 100)

    print(
        "File:",
        path.relative_to(PROJECT_ROOT)
    )

    print(
        "Exists:",
        path.exists()
    )

    if path.exists():

        df = pd.read_csv(path)

        evaluation_data[name] = df

        print(
            "Shape:",
            df.shape
        )

        print(
            "Columns:"
        )

        for column in df.columns:
            print(
                f"  - {column}"
            )

        print("\nPreview:")

        display(
            df.head()
        )

PRIMARY EVALUATION ARTIFACT INSPECTION

SVD Component Search
----------------------------------------------------------------------------------------------------
File: data/processed/svd_component_search.csv
Exists: True
Shape: (14, 3)
Columns:
  - representation
  - components
  - explained_variance_pct

Preview:


,representation,components,explained_variance_pct
0,Core,10,8.767354
1,Core,25,16.685701
2,Core,50,24.194685
3,Core,75,29.797957
4,Core,100,34.661219



SVD Neighbor Preservation
----------------------------------------------------------------------------------------------------
File: data/processed/svd_neighbor_preservation.csv
Exists: True
Shape: (6, 3)
Columns:
  - representation
  - components
  - top10_neighbor_preservation_pct

Preview:


,representation,components,top10_neighbor_preservation_pct
0,Core,100,47.800000
1,Core,150,50.833333
2,Core,200,52.566667
3,Enriched,100,50.566667
4,Enriched,150,55.033333



SVD Similarity Preservation
----------------------------------------------------------------------------------------------------
File: data/processed/svd_similarity_preservation.csv
Exists: True
Shape: (6, 5)
Columns:
  - representation
  - components
  - explained_variance_pct
  - similarity_spearman_rho
  - p_value

Preview:


...

,query,recommendations,unique_books,mean_similarity,max_similarity,unique_clusters,dominant_cluster,dominant_cluster_count,dominant_cluster_pct
0,New Manager,10,10,0.239194,0.475328,5,Organizational Culture and Trust,5,50.0
1,Conflict Management,10,10,0.194382,0.314164,5,Organizational Culture and Trust,4,40.0
2,Strategy,10,10,0.281884,0.361138,4,Decision Making,6,60.0
3,Organizational Change,10,10,0.239024,0.364433,4,Change Management,7,70.0
4,Entrepreneurship,10,10,0.239672,0.318541,3,Entrepreneurial and Growth Mindset,7,70.0



LLM Retrieval Robustness
----------------------------------------------------------------------------------------------------
File: data/processed/llm_retrieval_robustness.csv
Exists: True
Shape: (4, 6)
Columns:
  - test
  - query
  - recommendations
  - unique_books
  - all_positive_similarity
  - top_result

Preview:


,test,query,recommendations,unique_books,all_positive_similarity,top_result
0,Empty Query,NaN,0,0,NaN,NaN
1,Out of Vocabulary,qxzvplm jjjkkk zzzqqq,0,0,NaN,NaN
2,Focused Query,negotiation and conflict resolution,5,5,True,The Seven Tensions of Negotiation
3,Broad Query,leadership management,5,5,True,Leadership Development Studies



Cluster Summary
----------------------------------------------------------------------------------------------------
File: data/processed/final_topic_cluster_summary.csv
Exists: True
Shape: (29, 10)
Columns:
  - cluster
  - cluster_label
  - cluster_size
  - top_terms
  - representative_titles
  - average_centroid_distance
  - mean_distance_to_centroid
  - median_distance_to_centroid
  - max_distance_to_centroid
  - cohesion_group

Preview:


,cluster,cluster_label,cluster_size,top_terms,representative_titles,average_centroid_distance,mean_distance_to_centroid,median_distance_to_centroid,max_distance_to_centroid,cohesion_group
0,0,Entrepreneurial and Growth Mindset,82,"mindset, art, new, advantage, startup, entrepr...",A Platform Mindset | Unstoppable Mindset | The...,0.409,0.794508,0.836619,1.000000,Broadest
1,1,Organizational Behavior,56,"behavior, organizational behavior, organizatio...",Organizational Behavior | Organizational behav...,0.093,0.396443,0.413353,0.693245,Moderately compact
2,2,Team Leadership,33,"team, team leadership, leadership, effective t...",Team leadership | Team Leadership | Team Leade...,0.118,0.446914,0.493052,0.710623,Moderately compact
3,3,Future Thinking and Personal Development,70,"future, think, self, excellence, war, want, st...",Stewards of the Future | Future Tense | Invent...,0.497,0.775839,0.802344,0.933148,Broadest
4,4,Decision Making,33,"decision, making, decision making, prise, judg...",DECISION MAKING | Creative decision making | J...,0.242,0.510835,0.460445,0.930248,Moderately broad


## 3. Consolidated Quantitative Evaluation

The primary quantitative results from the previous modeling stages are now consolidated into a single evaluation framework.

The purpose of this comparison is not to identify one universal "best model."

Each component performs a different function:

- SVD evaluates dimensionality reduction;
- clustering evaluates thematic structure;
- the recommender evaluates retrieval behavior;
- the autoencoder evaluates neural compression;
- natural-language retrieval evaluates query handling and robustness.

The consolidated metrics therefore provide evidence for deciding which component should be retained for each role in the final system.

In [4]:
# ============================================================
# CONSOLIDATED MODEL EVALUATION METRICS
# ============================================================

# ------------------------------------------------------------
# Retrieve loaded evaluation tables
# ------------------------------------------------------------

svd_similarity = evaluation_data[
    "SVD Similarity Preservation"
]

svd_neighbors = evaluation_data[
    "SVD Neighbor Preservation"
]

rec_similarity = evaluation_data[
    "Recommendation Evaluation"
]

rec_diversity = evaluation_data[
    "Recommendation Diversity"
]

rec_exposure = evaluation_data[
    "Recommendation Source Exposure"
]

autoencoder_eval = evaluation_data[
    "Autoencoder Evaluation"
]

llm_diagnostic = evaluation_data[
    "LLM Retrieval Diagnostic"
]

llm_robustness = evaluation_data[
    "LLM Retrieval Robustness"
]

cluster_summary = evaluation_data[
    "Cluster Summary"
]


# ------------------------------------------------------------
# 1. SVD — selected 200-component representations
# ------------------------------------------------------------

svd_200 = (
    svd_similarity[
        svd_similarity["components"] == 200
    ]
    .merge(
        svd_neighbors[
            svd_neighbors["components"] == 200
        ],
        on=[
            "representation",
            "components"
        ],
        how="left"
    )
)


# ------------------------------------------------------------
# 2. Recommendation metrics
# ------------------------------------------------------------

recommendation_metrics = {
    "Recommendation pairs":
        len(rec_similarity),

    "Eligible query books":
        rec_diversity["query_book_id"].nunique(),

    "Mean similarity":
        rec_similarity["similarity_score"].mean(),

    "Median similarity":
        rec_similarity["similarity_score"].median(),

    "Mean Rank-1 similarity":
        rec_similarity.loc[
            rec_similarity[
                "recommendation_rank"
            ] == 1,
            "similarity_score"
        ].mean(),

    "Mean Rank-10 similarity":
        rec_similarity.loc[
            rec_similarity[
                "recommendation_rank"
            ] == 10,
            "similarity_score"
        ].mean(),

    "Mean unique topic clusters":
        rec_diversity[
            "unique_topic_clusters"
        ].mean(),

    "Median unique topic clusters":
        rec_diversity[
            "unique_topic_clusters"
        ].median(),

    "Mean cluster diversity ratio":
        rec_diversity[
            "cluster_diversity_ratio"
        ].mean()
}


# ------------------------------------------------------------
# 3. Catalog coverage
# ------------------------------------------------------------

catalog_size = 2067

unique_recommended_books = (
    rec_similarity[
        "recommended_book_id"
    ]
    .nunique()
)

catalog_coverage_pct = (
    unique_recommended_books
    / catalog_size
    * 100
)

recommendation_metrics[
    "Unique books recommended"
] = unique_recommended_books

recommendation_metrics[
    "Catalog coverage (%)"
] = catalog_coverage_pct


# ------------------------------------------------------------
# 4. Autoencoder metrics
# ------------------------------------------------------------

autoencoder_metrics = (
    autoencoder_eval
    .set_index("metric")["value"]
    .to_dict()
)


# ------------------------------------------------------------
# 5. Clustering metrics
# ------------------------------------------------------------

cluster_metrics = {
    "Number of clusters":
        cluster_summary[
            "cluster"
        ].nunique(),

    "Books represented":
        int(
            cluster_summary[
                "cluster_size"
            ].sum()
        ),

    "Mean cluster size":
        cluster_summary[
            "cluster_size"
        ].mean(),

    "Median cluster size":
        cluster_summary[
            "cluster_size"
        ].median(),

    "Mean centroid distance":
        cluster_summary[
            "mean_distance_to_centroid"
        ].mean()
}


# ------------------------------------------------------------
# 6. Natural-language retrieval metrics
# ------------------------------------------------------------

llm_metrics = {
    "Diagnostic queries":
        len(llm_diagnostic),

    "Mean recommendations per diagnostic":
        llm_diagnostic[
            "recommendations"
        ].mean(),

    "Mean unique books per diagnostic":
        llm_diagnostic[
            "unique_books"
        ].mean(),

    "Mean query similarity":
        llm_diagnostic[
            "mean_similarity"
        ].mean(),

    "Mean maximum similarity":
        llm_diagnostic[
            "max_similarity"
        ].mean(),

    "Robustness tests":
        len(llm_robustness)
}


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("SELECTED SVD — 200 COMPONENTS")
print("=" * 80)

display(
    svd_200
)


print("\nRECOMMENDATION SYSTEM")
print("=" * 80)

display(
    pd.DataFrame(
        recommendation_metrics.items(),
        columns=[
            "metric",
            "value"
        ]
    )
)


print("\nAUTOENCODER")
print("=" * 80)

display(
    pd.DataFrame(
        autoencoder_metrics.items(),
        columns=[
            "metric",
            "value"
        ]
    )
)


print("\nTOPIC CLUSTERING")
print("=" * 80)

display(
    pd.DataFrame(
        cluster_metrics.items(),
        columns=[
            "metric",
            "value"
        ]
    )
)


print("\nNATURAL-LANGUAGE RETRIEVAL")
print("=" * 80)

display(
    pd.DataFrame(
        llm_metrics.items(),
        columns=[
            "metric",
            "value"
        ]
    )
)


print("\nSOURCE EXPOSURE")
print("=" * 80)

display(
    rec_exposure
)

SELECTED SVD — 200 COMPONENTS


,representation,components,explained_variance_pct,similarity_spearman_rho,p_value,top10_neighbor_preservation_pct
0,Core,200,49.432955,0.387264,0.0,52.566667
1,Enriched,200,41.948703,0.515777,0.0,57.966667



RECOMMENDATION SYSTEM


,metric,value
0,Recommendation pairs,19396.000000
1,Eligible query books,2040.000000
2,Mean similarity,0.321981
3,Median similarity,0.301556
4,Mean Rank-1 similarity,0.513143
5,Mean Rank-10 similarity,0.233117
6,Mean unique topic clusters,3.840686
7,Median unique topic clusters,4.000000
8,Mean cluster diversity ratio,0.467305
9,Unique books recommended,2016.000000



AUTOENCODER


...

,metric,value
0,Number of clusters,29.000000
1,Books represented,1884.000000
2,Mean cluster size,64.965517
3,Median cluster size,47.000000
4,Mean centroid distance,0.509168



NATURAL-LANGUAGE RETRIEVAL


,metric,value
0,Diagnostic queries,6.000000
1,Mean recommendations per diagnostic,10.000000
2,Mean unique books per diagnostic,10.000000
3,Mean query similarity,0.233811
4,Mean maximum similarity,0.364390
5,Robustness tests,4.000000



SOURCE EXPOSURE


,source_group,catalog_books,catalog_pct,recommendation_count,recommendation_pct,exposure_difference_pct_points
0,Both,3,0.145138,22,0.113425,-0.031712
1,LeadershipNow only,1117,54.039671,8774,45.236131,-8.803540
2,Open Library only,947,45.815191,10600,54.650443,8.835252


## 4. Model-by-Model Evaluation and Architecture Selection

The consolidated results demonstrate that the project's models should not be treated as competing algorithms for the same task.

Instead, each model provides a different capability within the final recommendation architecture.

The final selection is therefore based on **fitness for purpose**.

---

### 4.1 Enriched TF-IDF — Production Recommendation Representation

The enriched TF-IDF representation remains the primary representation for the production recommendation system.

The content-based recommender generated:

- **19,396 recommendation pairs**
- **2,040 eligible query books**
- **2,016 unique books recommended**
- **97.53% catalog coverage**
- mean cosine similarity of **0.322**
- median cosine similarity of **0.302**
- mean Rank-1 similarity of **0.513**
- mean Rank-10 similarity of **0.233**

Recommendation similarity decreases from the highest-ranked recommendations toward Rank 10, which is consistent with cosine-similarity ranking.

Topic diversity was also present within recommendation lists:

- mean unique topic clusters: **3.84**
- median unique topic clusters: **4**
- mean cluster diversity ratio: **0.467**

These results indicate that the system achieves broad catalog exposure while retrieving books with positive textual similarity.

The enriched TF-IDF representation is therefore retained for:

- book-to-book recommendation;
- natural-language query retrieval;
- cosine-similarity ranking.

It remains preferable to the reduced representations for production retrieval because it preserves the original sparse feature space rather than approximating similarity after dimensionality reduction.

---

### 4.2 Truncated SVD / LSA — Dimensionality Reduction

At 200 components, the Core and Enriched representations produced different trade-offs.

| Representation | Explained Variance | Similarity Spearman ρ | Top-10 Neighbor Preservation |
|---|---:|---:|---:|
| Core | 49.43% | 0.387 | 52.57% |
| Enriched | 41.95% | 0.516 | 57.97% |

Core SVD explains more total variance.

However, Enriched SVD provides stronger preservation of the original similarity structure and better Top-10 neighbor preservation.

This distinction is important because explained variance alone does not determine suitability for semantic applications.

The **200-dimensional Enriched LSA representation** is therefore retained for:

- dense semantic representation;
- dimensionality reduction;
- clustering support;
- visualization;
- PyTorch experimentation.

It does **not** replace the original enriched TF-IDF matrix for exact recommendation ranking.

---

### 4.3 Topic Clustering — Supplementary Semantic Structure

The final clustering solution contains:

- **29 topic clusters**
- **1,884 clustered books**
- mean cluster size of approximately **64.97 books**
- median cluster size of **47 books**

The clusters provide interpretable thematic organization across the catalog.

Examples identified during cluster interpretation include topics such as:

- Team Leadership;
- Decision Making;
- Change Management;
- Organizational Behavior;
- Entrepreneurial and Growth Mindset;
- Transformational Leadership.

Cluster cohesion is heterogeneous.

Some clusters are relatively compact, while others contain broader semantic themes.

For this reason, topic clusters are not used as hard recommendation constraints.

Instead, they provide:

- supplementary recommendation context;
- topic labels;
- exploratory navigation;
- recommendation-diversity evaluation;
- potential filtering and visualization features.

The clustering model therefore complements the recommendation system rather than determining recommendation ranking.

---

### 4.4 PyTorch Autoencoder — Experimental Neural Compression

The autoencoder compressed the 200-dimensional LSA representation into a 32-dimensional latent representation.

Reconstruction performance improved substantially:

- untrained test MSE: **0.006692**
- trained test MSE: **0.001188**
- test MSE improvement: **82.25%**

This demonstrates that the neural network successfully learned to reconstruct the LSA input representation.

However, reconstruction quality alone is not sufficient for a recommendation system.

Semantic-structure evaluation produced:

- pairwise cosine-similarity Spearman ρ: **0.107**
- mean shared Top-10 neighbors: **4.27 out of 10**
- median shared Top-10 neighbors: **4 out of 10**
- mean Top-10 neighbor preservation: **42.75%**
- median Top-10 neighbor preservation: **40%**

The very small p-value associated with the Spearman correlation indicates statistical evidence of an association, but the effect size itself is weak.

Therefore, the 32-dimensional autoencoder representation does not preserve the original semantic geometry sufficiently well to replace the TF-IDF recommendation representation.

The autoencoder is retained as an **experimental neural-network component**, demonstrating:

- unsupervised representation learning;
- PyTorch implementation;
- nonlinear dimensionality reduction;
- reconstruction evaluation;
- semantic-preservation evaluation.

It is **not selected as the production recommendation model**.

This conclusion applies to the tested architecture and 32-dimensional latent representation and should not be generalized to all autoencoder architectures.

---

### 4.5 Natural-Language Retrieval

The natural-language retrieval extension was evaluated using six different leadership and management scenarios.

Across these diagnostic queries:

- mean recommendations returned: **10**
- mean unique books returned: **10**
- mean recommendation similarity: **0.234**
- mean maximum similarity: **0.364**

The system differentiated between several user intents, including:

- new-manager development;
- conflict management;
- strategy;
- organizational change;
- entrepreneurship;
- employee motivation.

Four additional robustness tests demonstrated that:

- empty input returns no recommendations;
- out-of-vocabulary input returns no recommendations;
- focused valid queries return positive-similarity recommendations;
- broad valid queries continue to return valid recommendations.

The system therefore avoids arbitrary recommendation padding when textual evidence is insufficient.

Natural-language retrieval is retained as the primary query interface for the final application.

---

### 4.6 Source Exposure

Recommendation exposure differs from the source composition of the catalog.

| Source | Catalog Share | Recommendation Share | Difference |
|---|---:|---:|---:|
| Both | 0.15% | 0.11% | -0.03 pp |
| LeadershipNow only | 54.04% | 45.24% | -8.80 pp |
| Open Library only | 45.82% | 54.65% | +8.84 pp |

Open Library books therefore receive greater recommendation exposure relative to their catalog share, while LeadershipNow books receive lower exposure.

Previous controlled Core-versus-Enriched analysis also showed that source-associated recommendation behavior exists even in the Core title-and-author representation and becomes stronger with metadata enrichment.

This pattern should therefore be described as **source-associated recommendation behavior**, not automatically as algorithmic bias.

A plausible contributing factor is metadata heterogeneity between the two sources.

The limitation should remain visible in the final application and project conclusions.

---

### 4.7 LLM Integration

The LLM component is not selected as a recommendation engine.

Instead, the architecture follows:

**User Query**

→ **Enriched TF-IDF Retrieval**

→ **Cosine-Similarity Ranking**

→ **Structured Grounding**

→ **Optional LLM Explanation**

The LLM therefore cannot independently determine which books enter the recommendation list.

The grounding prompt requires:

- preservation of recommendation ranking;
- use only of retrieved books;
- no fabricated metadata;
- explicit treatment of missing information;
- correct interpretation of similarity scores.

No live LLM inference was executed during the current experiment because no external API credential was configured.

The project therefore demonstrates an **LLM-ready grounded architecture**, rather than claiming evaluated live LLM performance.

## 5. Final Architecture Decision

The evaluation supports a hybrid architecture in which different models perform specialized functions.

### Production Architecture

**User Input**

↓  

**Natural-Language Query or Existing Book**

↓  

**Enriched TF-IDF Representation**

↓  

**Cosine Similarity**

↓  

**Positive-Similarity Filtering**

↓  

**Duplicate Suppression**

↓  

**Ranked Top-N Recommendations**

↓  

**Topic-Cluster Metadata**

↓  

**Structured Grounding**

↓  

**Offline Presentation or Optional LLM Explanation**

---

### Supporting Analytical Components

**Enriched LSA — 200 dimensions**

Used for:

- dimensionality reduction;
- dense semantic representation;
- clustering;
- visualization;
- neural-network experimentation.

**K-Means — 29 topic clusters**

Used for:

- topic interpretation;
- recommendation context;
- diversity analysis;
- exploratory navigation.

**PyTorch Autoencoder — 32-dimensional latent space**

Used as:

- an experimental neural representation;
- evidence comparing reconstruction quality with semantic preservation.

It is not used for production recommendation ranking.

---

### Final Model Selection

The final production recommendation engine is:

**Enriched TF-IDF + Cosine Similarity**

The final supporting semantic representation is:

**200-Dimensional Enriched LSA**

The final topic-discovery model is:

**29-Cluster K-Means Solution**

The neural network remains:

**Experimental Autoencoder**

The generative-AI component remains:

**Optional Grounded Explanation Layer**

This architecture is selected because it provides the strongest combination of:

- recommendation coverage;
- semantic relevance;
- interpretability;
- methodological transparency;
- computational practicality;
- deployment readiness.

## 6. Final Model Evaluation Summary

The principal modeling components are summarized below according to their intended function, quantitative evidence, limitations, and final role in the system.

The table does not rank the models against one another because they address different analytical tasks.

Instead, it records the evidence supporting the final architecture decision.

In [5]:
# ============================================================
# FINAL MODEL EVALUATION SUMMARY TABLE
# ============================================================

model_evaluation_summary = pd.DataFrame([
    {
        "component": "Enriched TF-IDF + Cosine",
        "primary_function": "Production recommendation retrieval",
        "key_metric": "Catalog coverage",
        "metric_value": 97.532656,
        "metric_unit": "%",
        "supporting_evidence":
            "2,016 unique books recommended; "
            "mean similarity 0.322; "
            "mean Rank-1 similarity 0.513",
        "main_limitation":
            "Lexical sensitivity and heterogeneous metadata",
        "final_role": "Production"
    },

    {
        "component": "Enriched SVD / LSA (200D)",
        "primary_function":
            "Dense semantic representation",
        "key_metric":
            "Top-10 neighbor preservation",
        "metric_value": 57.966667,
        "metric_unit": "%",
        "supporting_evidence":
            "Similarity Spearman rho 0.516; "
            "41.95% explained variance",
        "main_limitation":
            "Reduced representation approximates original "
            "TF-IDF geometry",
        "final_role": "Supporting"
    },

    {
        "component": "K-Means (29 clusters)",
        "primary_function":
            "Topic discovery and semantic organization",
        "key_metric":
            "Books clustered",
        "metric_value": 1884,
        "metric_unit": "books",
        "supporting_evidence":
            "29 interpretable topic clusters; "
            "median cluster size 47",
        "main_limitation":
            "Cluster cohesion varies across topics",
        "final_role": "Supporting"
    },

    {
        "component": "Autoencoder (200→32→200)",
        "primary_function":
            "Experimental nonlinear compression",
        "key_metric":
            "Test MSE improvement",
        "metric_value": 82.247979,
        "metric_unit": "%",
        "supporting_evidence":
            "Final test MSE 0.001188; "
            "semantic rho 0.107; "
            "Top-10 preservation 42.75%",
        "main_limitation":
            "Weak preservation of original semantic geometry",
        "final_role": "Experimental"
    },

    {
        "component": "Natural-Language Retrieval",
        "primary_function":
            "User-intent query interface",
        "key_metric":
            "Successful diagnostic retrieval",
        "metric_value": 6,
        "metric_unit": "queries",
        "supporting_evidence":
            "10 unique recommendations per diagnostic; "
            "empty/OOV queries safely return zero results",
        "main_limitation":
            "Broad and multi-concept queries can be less specific",
        "final_role": "Production Interface"
    },

    {
        "component": "Grounded LLM Layer",
        "primary_function":
            "Recommendation explanation",
        "key_metric":
            "Grounding catalog coverage",
        "metric_value": 2067,
        "metric_unit": "books",
        "supporting_evidence":
            "Ranking-preserving prompt; "
            "anti-fabrication rules; "
            "provider-independent architecture",
        "main_limitation":
            "No live LLM inference evaluated",
        "final_role": "Optional Explanation"
    }
])


print("FINAL MODEL EVALUATION SUMMARY")
print("=" * 120)

display(
    model_evaluation_summary
)

FINAL MODEL EVALUATION SUMMARY


,component,primary_function,key_metric,metric_value,metric_unit,supporting_evidence,main_limitation,final_role
0,Enriched TF-IDF + Cosine,Production recommendation retrieval,Catalog coverage,97.532656,%,"2,016 unique books recommended; mean similarit...",Lexical sensitivity and heterogeneous metadata,Production
1,Enriched SVD / LSA (200D),Dense semantic representation,Top-10 neighbor preservation,57.966667,%,Similarity Spearman rho 0.516; 41.95% explaine...,Reduced representation approximates original T...,Supporting
2,K-Means (29 clusters),Topic discovery and semantic organization,Books clustered,1884.000000,books,29 interpretable topic clusters; median cluste...,Cluster cohesion varies across topics,Supporting
3,Autoencoder (200→32→200),Experimental nonlinear compression,Test MSE improvement,82.247979,%,Final test MSE 0.001188; semantic rho 0.107; T...,Weak preservation of original semantic geometry,Experimental
4,Natural-Language Retrieval,User-intent query interface,Successful diagnostic retrieval,6.000000,queries,10 unique recommendations per diagnostic; empt...,Broad and multi-concept queries can be less sp...,Production Interface
5,Grounded LLM Layer,Recommendation explanation,Grounding catalog coverage,2067.000000,books,Ranking-preserving prompt; anti-fabrication ru...,No live LLM inference evaluated,Optional Explanation


## 7. Save Final Model Evaluation Artifacts

The consolidated evaluation results and final architecture decision are saved as reusable project artifacts.

These outputs provide a traceable connection between the individual modeling notebooks and the final deployed recommendation architecture.

The saved evaluation artifacts can also support:

- project documentation;
- Tableau visualization;
- Streamlit presentation;
- final capstone reporting;
- presentation slides.

No trained models are modified during this stage.

In [6]:
# ============================================================
# SAVE FINAL MODEL EVALUATION ARTIFACTS
# ============================================================

import json


# ------------------------------------------------------------
# Output paths
# ------------------------------------------------------------

MODEL_EVALUATION_PATH = (
    PROCESSED_DIR
    / "final_model_evaluation_summary.csv"
)

ARCHITECTURE_PATH = (
    MODELS_DIR
    / "final_model_architecture.json"
)


# ------------------------------------------------------------
# Save evaluation summary
# ------------------------------------------------------------

model_evaluation_summary.to_csv(
    MODEL_EVALUATION_PATH,
    index=False
)


# ------------------------------------------------------------
# Final architecture decision
# ------------------------------------------------------------

final_architecture = {

    "production_recommender": {
        "representation":
            "Enriched TF-IDF",

        "ranking_method":
            "Cosine Similarity",

        "catalog_size":
            2067,

        "eligible_books":
            2040,

        "catalog_coverage_pct":
            97.532656
    },

    "semantic_representation": {
        "method":
            "Truncated SVD / LSA",

        "representation":
            "Enriched",

        "dimensions":
            200,

        "similarity_spearman_rho":
            0.515777,

        "top10_neighbor_preservation_pct":
            57.966667
    },

    "topic_model": {
        "method":
            "K-Means",

        "clusters":
            29,

        "books_clustered":
            1884,

        "role":
            "Supplementary topic metadata"
    },

    "neural_network": {
        "method":
            "Autoencoder",

        "architecture":
            "200-128-64-32-64-128-200",

        "latent_dimensions":
            32,

        "test_mse_improvement_pct":
            82.247979,

        "similarity_spearman_rho":
            0.1074673,

        "top10_neighbor_preservation_pct":
            42.74510,

        "role":
            "Experimental"
    },

    "natural_language_retrieval": {
        "enabled":
            True,

        "representation":
            "Enriched TF-IDF",

        "ranking_method":
            "Cosine Similarity",

        "diagnostic_queries":
            6,

        "robustness_tests":
            4
    },

    "llm_layer": {
        "mode":
            "offline_grounding",

        "live_api_enabled":
            False,

        "role":
            "Optional grounded explanation",

        "grounding_books":
            2067
    }
}


with open(
    ARCHITECTURE_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        final_architecture,
        file,
        indent=4
    )


print("FINAL EVALUATION ARTIFACTS SAVED")
print("=" * 80)

print(
    "Evaluation summary:",
    MODEL_EVALUATION_PATH
)

print(
    "Architecture decision:",
    ARCHITECTURE_PATH
)

FINAL EVALUATION ARTIFACTS SAVED
Evaluation summary: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/final_model_evaluation_summary.csv
Architecture decision: /Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/models/final_model_architecture.json


## 8. Final Integrity Validation

The saved evaluation artifacts are reloaded and checked against the validated results from the previous modeling stages.

The validation confirms that:

- all major modeling components are represented;
- the selected production recommender is correctly recorded;
- the 200-dimensional enriched LSA representation is retained;
- the 29-cluster topic model is correctly recorded;
- the autoencoder remains experimental;
- natural-language retrieval remains enabled;
- the LLM layer remains in offline grounding mode.

This final check ensures that the architecture recorded for deployment is consistent with the empirical evaluation.

In [7]:
# ============================================================
# FINAL MODEL EVALUATION INTEGRITY VALIDATION
# ============================================================

# Reload evaluation summary
evaluation_reload = pd.read_csv(
    MODEL_EVALUATION_PATH
)


# Reload architecture
with open(
    ARCHITECTURE_PATH,
    "r",
    encoding="utf-8"
) as file:

    architecture_reload = json.load(
        file
    )


# ------------------------------------------------------------
# Validation checks
# ------------------------------------------------------------

validation_checks = {

    "Evaluation artifact exists":
        MODEL_EVALUATION_PATH.exists(),

    "Architecture artifact exists":
        ARCHITECTURE_PATH.exists(),

    "Six components recorded":
        len(evaluation_reload) == 6,

    "Production recommender is Enriched TF-IDF":
        architecture_reload[
            "production_recommender"
        ][
            "representation"
        ] == "Enriched TF-IDF",

    "Ranking uses cosine similarity":
        architecture_reload[
            "production_recommender"
        ][
            "ranking_method"
        ] == "Cosine Similarity",

    "Catalog size is 2067":
        architecture_reload[
            "production_recommender"
        ][
            "catalog_size"
        ] == 2067,

    "Eligible books are 2040":
        architecture_reload[
            "production_recommender"
        ][
            "eligible_books"
        ] == 2040,

    "LSA dimensions are 200":
        architecture_reload[
            "semantic_representation"
        ][
            "dimensions"
        ] == 200,

    "Topic clusters are 29":
        architecture_reload[
            "topic_model"
        ][
            "clusters"
        ] == 29,

    "Autoencoder latent dimensions are 32":
        architecture_reload[
            "neural_network"
        ][
            "latent_dimensions"
        ] == 32,

    "Autoencoder role is Experimental":
        architecture_reload[
            "neural_network"
        ][
            "role"
        ] == "Experimental",

    "Natural-language retrieval enabled":
        architecture_reload[
            "natural_language_retrieval"
        ][
            "enabled"
        ] is True,

    "LLM live API disabled":
        architecture_reload[
            "llm_layer"
        ][
            "live_api_enabled"
        ] is False,

    "LLM mode is offline grounding":
        architecture_reload[
            "llm_layer"
        ][
            "mode"
        ] == "offline_grounding"
}


validation_df = pd.DataFrame(
    validation_checks.items(),
    columns=[
        "validation_check",
        "passed"
    ]
)


print("FINAL MODEL EVALUATION VALIDATION")
print("=" * 100)

display(
    validation_df
)


print(
    "\nChecks passed:",
    int(
        validation_df[
            "passed"
        ].sum()
    ),
    "/",
    len(validation_df)
)


print(
    "All validation checks passed:",
    bool(
        validation_df[
            "passed"
        ].all()
    )
)

FINAL MODEL EVALUATION VALIDATION


,validation_check,passed
0,Evaluation artifact exists,True
1,Architecture artifact exists,True
2,Six components recorded,True
3,Production recommender is Enriched TF-IDF,True
4,Ranking uses cosine similarity,True
5,Catalog size is 2067,True
6,Eligible books are 2040,True
7,LSA dimensions are 200,True
8,Topic clusters are 29,True
9,Autoencoder latent dimensions are 32,True



Checks passed: 14 / 14
All validation checks passed: True


# Notebook 16 — Model Evaluation: Final Summary

## Objective

This notebook consolidated the quantitative and qualitative evaluation evidence generated throughout the Leadership and Management Book Recommendation System project.

Rather than treating all models as competitors for the same task, each component was evaluated according to its intended function.

The evaluation covered:

- TF-IDF content representation and recommendation;
- Truncated SVD / Latent Semantic Analysis;
- K-Means topic clustering;
- content-based recommendation;
- PyTorch autoencoder compression;
- natural-language retrieval;
- grounded LLM integration architecture.

---

## 1. Production Recommendation Model

The final production recommendation engine is:

**Enriched TF-IDF + Cosine Similarity**

The recommendation evaluation produced:

- Catalog size: **2,067 books**
- Recommendation-eligible books: **2,040**
- Recommendation pairs evaluated: **19,396**
- Unique books recommended: **2,016**
- Catalog coverage: **97.53%**
- Mean recommendation similarity: **0.322**
- Median recommendation similarity: **0.302**
- Mean Rank-1 similarity: **0.513**
- Mean Rank-10 similarity: **0.233**

Recommendation lists also contained thematic diversity:

- Mean unique topic clusters per query: **3.84**
- Median unique topic clusters: **4**
- Mean cluster diversity ratio: **0.467**

These results support the use of the original enriched sparse TF-IDF representation for production recommendation ranking.

---

## 2. Dimensionality Reduction

The selected dimensionality-reduction configuration is:

**Enriched Truncated SVD / LSA — 200 dimensions**

At 200 components:

### Core Representation

- Explained variance: **49.43%**
- Similarity Spearman ρ: **0.387**
- Top-10 neighbor preservation: **52.57%**

### Enriched Representation

- Explained variance: **41.95%**
- Similarity Spearman ρ: **0.516**
- Top-10 neighbor preservation: **57.97%**

Although the Core representation explains more variance, the Enriched representation better preserves the original similarity relationships and local neighborhood structure.

The 200-dimensional Enriched LSA representation is therefore retained for:

- dense semantic representation;
- clustering support;
- visualization;
- PyTorch experimentation.

It does not replace TF-IDF for exact recommendation ranking.

---

## 3. Topic Clustering

The final topic-clustering solution contains:

- **29 clusters**
- **1,884 clustered books**
- Mean cluster size: **64.97**
- Median cluster size: **47**
- Mean distance to centroid: **0.509**

The clusters provide useful thematic organization, but cohesion varies between clusters.

K-Means is therefore retained as a supporting model for:

- topic labeling;
- exploratory navigation;
- recommendation context;
- diversity analysis;
- visualization.

Topic clusters do not determine recommendation ranking.

---

## 4. Neural-Network Evaluation

The PyTorch autoencoder used the architecture:

**200 → 128 → 64 → 32 → 64 → 128 → 200**

The model successfully learned to reconstruct the 200-dimensional LSA representation.

### Reconstruction Performance

- Untrained test MSE: **0.006692**
- Final test MSE: **0.001188**
- Test MSE improvement: **82.25%**

However, semantic-structure preservation was substantially weaker:

- Pairwise similarity Spearman ρ: **0.107**
- Mean shared Top-10 neighbors: **4.27 / 10**
- Median shared Top-10 neighbors: **4 / 10**
- Mean Top-10 neighbor preservation: **42.75%**
- Median Top-10 neighbor preservation: **40%**

The autoencoder therefore demonstrates successful reconstruction but does not preserve the original semantic geometry sufficiently well to replace the production TF-IDF recommender.

Its final role is:

**Experimental neural representation**

This conclusion applies specifically to the tested architecture and latent dimensionality.

---

## 5. Natural-Language Retrieval

The recommendation system was extended to accept natural-language leadership and management requests.

Six diagnostic scenarios were evaluated.

Across these queries:

- Mean recommendations returned: **10**
- Mean unique recommendations: **10**
- Mean similarity: **0.234**
- Mean maximum similarity: **0.364**

The retrieval system responded to different intents including:

- new-manager development;
- conflict management;
- strategy;
- organizational change;
- entrepreneurship;
- employee motivation.

Four robustness tests were also completed.

The system correctly returned:

- **0 recommendations** for empty input;
- **0 recommendations** for out-of-vocabulary input;
- valid positive-similarity recommendations for focused queries;
- valid recommendations for broad leadership queries.

Natural-language retrieval is therefore retained as the primary user-query interface.

---

## 6. Source Exposure

Recommendation exposure is not proportional to source composition.

### LeadershipNow

- Catalog share: **54.04%**
- Recommendation share: **45.24%**
- Difference: **−8.80 percentage points**

### Open Library

- Catalog share: **45.82%**
- Recommendation share: **54.65%**
- Difference: **+8.84 percentage points**

This pattern is described as **source-associated recommendation behavior**.

It should not automatically be interpreted as algorithmic bias because the two data sources contain substantially different metadata structures and levels of enrichment.

The effect remains an important limitation of the recommendation system.

---

## 7. LLM Integration

The LLM layer is not used as the recommendation engine.

The architecture is:

**User Query**

→ **Enriched TF-IDF**

→ **Cosine Similarity**

→ **Ranked Recommendations**

→ **Structured Grounding**

→ **Optional LLM Explanation**

The grounding layer covers all:

**2,067 books**

and constrains future generative output to retrieved evidence.

No live LLM inference was evaluated because no external API credential was configured.

The final LLM configuration therefore remains:

- Provider: **None**
- Live API: **Disabled**
- Mode: **Offline Grounding**

The project demonstrates an **LLM-ready grounded architecture**, not evaluated live LLM performance.

---

## 8. Final Model Architecture

The final system uses specialized components rather than a single universal model.

### Production Recommender

**Enriched TF-IDF + Cosine Similarity**

Purpose:

- book-to-book recommendation;
- natural-language retrieval;
- exact recommendation ranking.

### Semantic Representation

**Enriched LSA — 200 dimensions**

Purpose:

- dimensionality reduction;
- dense representation;
- clustering;
- visualization;
- neural experimentation.

### Topic Discovery

**K-Means — 29 clusters**

Purpose:

- thematic organization;
- recommendation context;
- diversity analysis;
- exploratory navigation.

### Neural Network

**PyTorch Autoencoder — 32-dimensional bottleneck**

Purpose:

- experimental nonlinear compression;
- representation-learning analysis.

Status:

**Experimental — not used for production ranking**

### Generative-AI Layer

**Grounded LLM-ready architecture**

Purpose:

- optional natural-language explanation of already retrieved recommendations.

Status:

**Offline grounding mode**

---

## 9. Model-Selection Conclusion

The evaluation demonstrates that increasing model complexity does not necessarily improve recommendation suitability.

The autoencoder achieved strong reconstruction performance but preserved less semantic neighborhood structure than the selected LSA representation.

Likewise, dimensionality reduction is valuable for clustering and experimentation but introduces approximation relative to the original sparse TF-IDF representation.

For this reason, the final system retains the original enriched TF-IDF representation for recommendation ranking while assigning SVD, clustering, neural networks, and LLM integration to specialized supporting roles.

The final architecture prioritizes:

- semantic relevance;
- broad catalog coverage;
- interpretability;
- transparency;
- robustness;
- computational practicality;
- deployment readiness.

---

## 10. Saved Evaluation Artifacts

### Final Model Evaluation Summary

`data/processed/final_model_evaluation_summary.csv`

Contains the consolidated evaluation of the six major system components.

### Final Model Architecture

`models/final_model_architecture.json`

Contains the machine-readable architecture selected for the final application.

---

## Final Integrity Validation

A final architecture integrity test evaluated **14 conditions**.

Result:

**14 / 14 checks passed**

All validation checks returned:

**True**

This confirms that the saved deployment architecture is consistent with the validated results from the preceding modeling notebooks.

---

# Final Status

**Notebook 16 — Model Evaluation: COMPLETE**

The final production recommendation model is:

**Enriched TF-IDF + Cosine Similarity**

supported by:

**200-D Enriched LSA + 29-Cluster K-Means + Natural-Language Retrieval + Grounded Explanation Architecture**

with the PyTorch autoencoder retained as an experimental neural-network component.